# DeepGuard — DF40 Drive-native download and extraction

This notebook keeps the large DF40 dataset on Google Drive and avoids Colab's small `/content` disk. The official DF40 README provides the dataset through a Google Form / Google Drive distribution; after obtaining the official Drive folder link, paste it below and the notebook downloads directly to your persistent Drive workspace.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, os, json, time, subprocess, sys
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DF40=ROOT/'datasets/DF40'
ARCH=ROOT/'datasets/archives'
DF40.mkdir(parents=True,exist_ok=True); ARCH.mkdir(parents=True,exist_ok=True)
usage=shutil.disk_usage('/content/drive')
print('Drive total:',round(usage.total/1e12,2),'TB')
print('Drive free :',round(usage.free/1e12,2),'TB')
print('DF40 target:',DF40)


## 1. Official DF40 distribution

DF40's official repository states that the processed test data are about 93 GB and provides the official download through Google Drive/Baidu. The official repository also states that the dataset is CC BY-NC 4.0. We do not bypass that access mechanism.

Open the official DF40 repository, complete its dataset-access step if required, and copy the resulting **Google Drive folder URL**. Then paste it in the next cell.


In [ ]:
OFFICIAL_DF40_FOLDER_URL = ''  # paste the Google Drive folder URL obtained from the official DF40 distribution

if not OFFICIAL_DF40_FOLDER_URL:
    print('PAUSED: paste the official DF40 Google Drive folder URL into OFFICIAL_DF40_FOLDER_URL and rerun this cell.')
else:
    print('Official DF40 folder URL supplied.')


In [ ]:
if OFFICIAL_DF40_FOLDER_URL:
    subprocess.run([sys.executable,'-m','pip','install','-q','gdown'],check=True)
    print('gdown ready')


In [ ]:
if OFFICIAL_DF40_FOLDER_URL:
    # gdown downloads folder contents file-by-file; destination is Google Drive, not /content.
    cmd=[sys.executable,'-m','gdown','--folder',OFFICIAL_DF40_FOLDER_URL,'-O',str(DF40),'--remaining-ok']
    print('Starting Drive-native download...')
    result=subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError('gdown could not complete the official Drive-folder download. If Google Drive blocks automated access, download/copy the folder in the browser into DeepGuard/datasets/DF40 and rerun the verification cells.')
    print('Download step finished.')


In [ ]:
# Verification: count files and total bytes without copying anything to /content.
files=[p for p in DF40.rglob('*') if p.is_file()]
size=sum(p.stat().st_size for p in files)
print('Files:',len(files))
print('Dataset size:',round(size/1e9,2),'GB')
print('Target:',DF40)
(DF40/'_deepguard_dataset_manifest.json').write_text(json.dumps({'timestamp':time.time(),'files':len(files),'bytes':size,'source':'official DF40 Google Drive distribution'},indent=2))


## 2. If gdown is blocked

Google may block automated folder downloads or require an authenticated browser session. In that case, use the official Google Drive web interface to copy the dataset into `My Drive/DeepGuard/datasets/DF40`. No local Colab copy is needed. Then rerun the verification cell above.


In [ ]:
# Final structure check for the DF40 methods expected by DeepfakeBench_DF40.
expected=['fsgan','faceswap','simswap','inswap','blendface','uniface','mobileswap','e4s','facedancer','fomm','facevid2vid','wav2lip','MRAA','one_shot_free','pirender','tpsm','lia','danet','sadtalker','mcnet','heygen','VQGAN','StyleGAN2','StyleGAN3','StyleGANXL','sd2.1','ddim','PixArt','DiT','SiT','MidJourney','whichfaceisreal','stargan','starganv2','styleclip','e4e','CollabDiff']
found={p.name for p in DF40.iterdir() if p.is_dir()}
print('Top-level method folders found:',len(found))
missing=[x for x in expected if x not in found]
print('Missing expected folders:',missing[:20], '...' if len(missing)>20 else '')
